# irchroma — synthetic demo (IR → SR → colorized RGB)

**BAH 2026 · Problem Statement 10.** This notebook runs the end-to-end pipeline on a
tiny, CPU-only, *procedural* IR↔RGB sample — no satellite data, no credentials, no GPU.

Flow: a low-contrast single-channel **IR** tile is super-resolved by the Stage-1 guided
SR model, then the Stage-2 semantic-conditioned colorizer turns the SR'd IR into a
realistic **RGB** image (water→blue, vegetation→green), with an O(1) class→Lab color-LUT
clamp keeping colors class-faithful. Requires `torch` (install with the CPU wheel:
`pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu`).

In [ ]:
# Make the src/ layout importable when running from the repo root.
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "src")))

import torch
from irchroma.config import synthetic_demo_config, LULC_CLASSES
from irchroma.data import make_synthetic_sample

cfg = synthetic_demo_config()  # tiny, CPU-only config (scale=2, small nets)
print("TORCH:", torch.__version__, "| device:", cfg.train.device, "| SR scale:", cfg.model.scale)
print(f"{len(LULC_CLASSES)} LULC classes:", LULC_CLASSES)

In [ ]:
# Build one deterministic synthetic Sample (semantically-consistent IR/RGB pair).
batch = make_synthetic_sample(cfg, batch_size=1, seed=0)
for key in ("ir", "rgb", "guide", "semantic"):
    val = batch[key]
    print(f"{key:9s}:", None if val is None else tuple(val.shape), getattr(val, "dtype", None))

# Sanity: IR is the low-res input; RGB is scale x larger (the SR target).
ir_hw = batch["ir"].shape[-2:]
rgb_hw = batch["rgb"].shape[-2:]
print("IR grid:", tuple(ir_hw), "-> RGB grid:", tuple(rgb_hw), "(scale =", cfg.model.scale, ")")

In [ ]:
# Build the end-to-end pipeline and run IR -> (SR, colorized RGB).
# `build_pipeline` lives in irchroma.models.pipeline (the composed two-stage model).
from irchroma.models.pipeline import build_pipeline

pipeline = build_pipeline(cfg)
if hasattr(pipeline, "eval"):
    pipeline.eval()

with torch.no_grad():
    out = pipeline.forward(batch)  # -> PipelineOutput: rgb [B,3,Hs,Ws] in [0,1], sr, ...

rgb = out["rgb"]
sr = out["sr"]
print("SR  :", tuple(sr.shape), "| RGB :", tuple(rgb.shape))
print("RGB range: [%.3f, %.3f] (expected within [0, 1])" % (float(rgb.min()), float(rgb.max())))

In [ ]:
# Visualize: input IR (upsampled for display), ground-truth RGB, and the colorized output.
import matplotlib.pyplot as plt

def chw_to_hwc(t):
    return t[0].detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy()

ir_disp = torch.nn.functional.interpolate(
    batch["ir"], size=tuple(rgb.shape[-2:]), mode="nearest"
)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
axes[0].imshow(ir_disp[0, 0].cpu().numpy(), cmap="inferno"); axes[0].set_title("input IR (low-res)")
axes[1].imshow(chw_to_hwc(batch["rgb"])); axes[1].set_title("ground-truth RGB")
axes[2].imshow(chw_to_hwc(rgb)); axes[2].set_title("irchroma colorized")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## What just happened (IR → SR → colorized)

1. **Synthetic sample** — a blobby land-cover label map paints a high-res RGB target; the
   single-channel IR is a low-contrast, low-resolution physical function of the *same*
   labels (so a wrong color shows up as a class mismatch — the no-hallucination signal).
2. **Stage-1 guided super-resolution** upsamples the coarse IR by `scale`, borrowing real
   high-frequency structure from the co-registered HR guide band instead of inventing it.
3. **Stage-2 semantic-conditioned colorization** maps the SR'd IR to RGB via a
   Pix2PixHD + DDColor generator with SPADE land-cover conditioning, then an **O(1)
   class→CIE-Lab color-LUT** clamps chroma toward each class's palette (water→blue,
   trees→green) while leaving luminance free so SR texture survives.

The output is a `PipelineOutput` (`rgb [B,3,H·s,W·s]` in [0,1], `sr`, optional
`semantic_pred`/`uncertainty`) — the same contract used by training, evaluation, and the
tile server. See `ARCHITECTURE.md` and `docs/research/` for the full design.